# Problem 12 (100 points)

**Multi-head attention (MHA)** is a big breakthrough in AI. Based on its original form, there are many variants that improved it.

In this problem, you are asked to study multi-head attention and its variants: **Grouped Query Attention (GQA)** and **Multi-head Latent Attention (MLA)**.

We use the following notation in this problem.

- $B$: batch size. $b$: index of a sample.
- $L_1$: length of an attending sequence. $l_1$: index of a position in this sequence.
- $L_2$: length of a being attended sequence. $l_2$: index of a position in this sequence.
- $D$: dimension of a hidden state/token (we assume $D_1 = D_2 = D$ throughout).
- $H$: number of heads. $h$: index of a head.
- $d$: dimension of a query/key/value vector per head (we assume $D_{qk} = D_v = d$).
- We have $H \cdot d = D$.
- $G$: number of groups (for GQA), where $G | H$ (i.e., $G$ divides $H$).
- $r$: latent/compressed dimension (for MLA).

In [ ]:
# Run code in this cell

"""
DO NOT MAKE ANY CHANGE IN THIS CELL.
"""

import torch
import torch.nn as nn
import numpy as np

> WARNING !!!
>
- Beyond importing libraries/modules/classes/functions in the preceding cell, you are **NOT allowed to import anything else for the following purposes**:
    - **As a part of your final solution.** For instance, if a problem asks you to build a model without using sklearn but you use it, then you will not earn points.
    - **Temporarily import something to assist you to get a solution.** For instance, if a problem asks you to manually compute eigenvalues but you temporarily use `np.linalg.eig` to get an answer and then delete your code, then you violate the rule.

    **Rule of thumb:** Each part has its particular purpose to intentionally test you something. Do not attempt to find a shortcut to circumvent the rule.

## Part 1 (5 points, non-coding task)

**Do the following tasks (Reasoning is not required).**

For each hidden state at position $l_1$ in an attending sequence, $\mathbf{x}_{l_1} \in \mathbb{R}^{D}$, we project it into a query vector for head $h$ according to

$$\mathbf{q}_{l_1,h} = \mathbf{W}^{\mathbf{Q}}_h \mathbf{x}_{l_1} .$$

For each hidden state at position $l_2$ in a being attended sequence, $\mathbf{y}_{l_2} \in \mathbb{R}^{D}$, we project it into key and value vectors for head $h$:

$$\mathbf{k}_{l_2,h} = \mathbf{W}^{\mathbf{K}}_h \mathbf{y}_{l_2}, \qquad \mathbf{v}_{l_2,h} = \mathbf{W}^{\mathbf{V}}_h \mathbf{y}_{l_2} .$$

1. What is the shape of $\mathbf{W}^{\mathbf{Q}}_h$?
2. What is the shape of $\mathbf{W}^{\mathbf{K}}_h$?
3. What is the shape of $\mathbf{W}^{\mathbf{V}}_h$?
4. We concatenate all per-head matrices along axis 0 for $\mathbf{M} \in \{\mathbf{Q}, \mathbf{K}, \mathbf{V}\}$:
   $$\mathbf{W}^{\mathbf{M}} = \begin{bmatrix} \mathbf{W}^{\mathbf{M}}_0 \\ \mathbf{W}^{\mathbf{M}}_1 \\ \vdots \\ \mathbf{W}^{\mathbf{M}}_{H-1} \end{bmatrix} .$$
   What are the shapes of $\mathbf{W}^{\mathbf{Q}}$, $\mathbf{W}^{\mathbf{K}}$, and $\mathbf{W}^{\mathbf{V}}$?
5. Define $\mathbf{q}_{l_1} = \begin{bmatrix} \mathbf{q}_{l_1,0} \\ \vdots \\ \mathbf{q}_{l_1,H-1} \end{bmatrix}$. What is the shape of $\mathbf{q}_{l_1}$, and what is its relationship to $\mathbf{W}^{\mathbf{Q}}$ and $\mathbf{x}_{l_1}$?

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 2 (5 points, non-coding task)

**Do the following tasks (Reasoning is not required).**

Define the attention score at position $l_1$ attending to position $l_2$ for head $h$ as

$$\alpha_{h, l_1 l_2} = \text{Softmax}_{l_2} \left( \frac{\mathbf{q}_{l_1,h}^\top \mathbf{k}_{l_2,h}}{\sqrt{d}} \right) .$$

The pre-out-projection output vector for head $h$ at position $l_1$ is

$$\mathbf{o}_{h,l_1} = \sum_{l_2 = 0}^{L_2 - 1} \alpha_{h, l_1 l_2} \, \mathbf{v}_{l_2,h} .$$

We concatenate across heads: $\mathbf{o}_{l_1} = \begin{bmatrix} \mathbf{o}_{0,l_1} \\ \vdots \\ \mathbf{o}_{H-1,l_1} \end{bmatrix}$ and apply an output projection:

$$\mathbf{x}_{l_1}^{\text{out}} = \mathbf{W}^O \mathbf{o}_{l_1}, \quad \text{where} \quad \mathbf{W}^O = \begin{bmatrix} \mathbf{W}^O_0 & \mathbf{W}^O_1 & \cdots & \mathbf{W}^O_{H-1} \end{bmatrix} .$$

1. What is the shape of $\mathbf{o}_{h,l_1}$?
2. What is the shape of $\mathbf{o}_{l_1}$?
3. What is the shape of $\mathbf{W}^O_h$? Of $\mathbf{W}^O$?
4. What is the shape of $\mathbf{x}_{l_1}^{\text{out}}$?
5. How many total parameters (ignoring biases) does MHA have? Express in terms of $D$.

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 3 (5 points, non-coding task)

**Do the following tasks (Reasoning is required).**

We now consider the batched tensor operations. Given an attending sequence tensor $\mathbf{X} \in \mathbb{R}^{B \times L_1 \times D}$ and being attended sequence tensor $\mathbf{Y} \in \mathbb{R}^{B \times L_2 \times D}$.

In PyTorch, `nn.Linear(D, H*d, bias=False)` stores a weight matrix of shape $(H \cdot d, D)$. When applied to $\mathbf{X}$, the result is $\mathbf{X} \mathbf{W}^\top$.

1. After $\mathbf{Q} = \text{W\_Q}(\mathbf{X})$, what is the shape of $\mathbf{Q}$? (1 point)
2. We reshape $\mathbf{Q}$ from $(B, L_1, H \cdot d)$ to $(B, L_1, H, d)$ and then permute to $(B, H, L_1, d)$. Why must we move $H$ before $L_1$? (2 points)
3. After the permutation, we compute $\text{logits} = \mathbf{Q} \, \mathbf{K}^{\mathsf{T}} / \sqrt{d}$, which is a batched matrix multiplication over the $(B, H)$ dimensions. What is the shape of $\text{logits}$, and what does each of its four dimensions represent? (2 points)

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 4 (10 points, coding task)

**In this part, you are asked to build your own multi-head attention module that subclasses `nn.Module`.**

- For simplicity, we ignore any masking. That is, each position in an attending sequence attends to all positions in a being attended sequence.
- The class name is `MyMHA`.
- Attributes:
  - `D`: Dimension of a hidden state/token.
  - `d`: Dimension of a query/key/value vector per head.
  - `H`: Number of heads.
  - `W_Q`: A linear module whose weight is a query-projection matrix. Shape consistent with Part 1. No bias.
  - `W_K`: A linear module whose weight is a key-projection matrix. Shape consistent with Part 1. No bias.
  - `W_V`: A linear module whose weight is a value-projection matrix. Shape consistent with Part 1. No bias.
  - `W_O`: A linear module whose weight is an out-projection matrix. Shape consistent with Part 2. No bias.
- Method `__init__`:
  - Inputs: `D`, `d`, `H`.
  - Outputs: None.
  - What to do inside: Initialize attribute values.
- Method `forward`:
  - Inputs:
    - An attending sequence (tensor) with shape `(B, L_1, D)`.
    - A being attended sequence (tensor) with shape `(B, L_2, D)`.
  - Outputs:
    - Post-out-projection outputs with shape `(B, L_1, D)`.
  - What to do inside:
    - Compute the outputs.
    - After each operation, add a comment on the tensor shape.
    - **Do not use any loop.**

### WRITE YOUR SOLUTION HERE ###

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 5 (10 points, coding task)

**Do the following tasks.**

1. Create `MyMHA(D=64, d=16, H=4)`. (1 point)
2. Create random tensors `x` of shape `(2, 10, 64)` and `y` of shape `(2, 20, 64)`. (1 point)
3. Compute `output = model(x, y)`. Assert `output.shape == (2, 10, 64)`. (2 points)
4. Count the total number of parameters. Show it equals $4 D^2$ (since $H \cdot d = D$ gives four $D \times D$ projection matrices). Print the count. (3 points)
5. Verify self-attention works: pass `x` as both inputs, i.e., `model(x, x)`. Assert the output shape is `(2, 10, 64)`. (3 points)

### WRITE YOUR SOLUTION HERE ###

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Next, let us study a variant of MHA: **Grouped Query Attention (GQA)**.

Recall that in MHA, the number of heads in queries, keys, and values are the same, $H$. Thus, query $\mathbf{q}_{l_1, h}$ attends to key $\mathbf{k}_{l_2, h}$ with the same head index $h$.

In GQA, we relax this constraint by allowing keys and values to have $G$ heads ($G \le H$), where $G$ is a factor of $H$. Each group of $H/G$ query heads shares one set of key/value projections.

Specifically, a query head ${\color{red}{h}}$ is associated with key/value group ${\color{blue}{g}}$ where

$${\color{blue}{g}} = \left\lfloor \frac{\color{red}{h}}{H/G} \right\rfloor .$$

As an example, suppose $H = 12$ and $G = 3$. Then:
- Group ${\color{blue}{g}} = 0$: query heads ${\color{red}{h}} = 0, 1, 2, 3$.
- Group ${\color{blue}{g}} = 1$: query heads ${\color{red}{h}} = 4, 5, 6, 7$.
- Group ${\color{blue}{g}} = 2$: query heads ${\color{red}{h}} = 8, 9, 10, 11$.

## Part 6 (5 points, non-coding task)

**Do the following tasks (Reasoning is required).**

For $\mathbf{M} \in \left\{ \mathbf{K}, \mathbf{V} \right\}$, the GQA projection matrix is

$$\mathbf{W}^{\mathbf{M}, \text{GQA}} = \begin{bmatrix} \mathbf{W}^{\mathbf{M}, \text{GQA}}_0 \\ \vdots \\ \mathbf{W}^{\mathbf{M}, \text{GQA}}_{G-1} \end{bmatrix} \in \mathbb{R}^{G \cdot d \times D} .$$

We "unroll" this by concatenating $H/G$ copies along axis 0:

$$\mathbf{\tilde W}^{\mathbf{M}, \text{GQA}} = \begin{bmatrix} \mathbf{W}^{\mathbf{M}, \text{GQA}} \\ \mathbf{W}^{\mathbf{M}, \text{GQA}} \\ \vdots \\ \mathbf{W}^{\mathbf{M}, \text{GQA}} \end{bmatrix} \in \mathbb{R}^{D \times D} .$$

1. What is the relationship between $\text{rank}\left(\mathbf{\tilde W}^{\mathbf{M}, \text{GQA}}\right)$ and $\text{rank}\left(\mathbf{W}^{\mathbf{M}, \text{GQA}}\right)$? Prove your answer. (3 points)
2. MHA is a special case of GQA. Explain why. (2 points)

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 7 (5 points, non-coding task)

**Do the following tasks (Reasoning is not required).**

Describe the tensor reshaping strategy for implementing GQA **without loops**, using broadcasting.

Given concrete values $B = 2, L = 10, H = 8, G = 2, d = 16$:

1. After projecting queries: $\mathbf{Q} = \text{W\_Q}(\mathbf{X})$ has shape $(B, L, H \cdot d)$. Write the concrete shape. Reshape to expose the group structure: $(B, L, G, H/G, d)$. Write the concrete shape. Then permute to $(B, G, H/G, L, d)$. (2 points)
2. After projecting keys: $\mathbf{K} = \text{W\_K}(\mathbf{X})$ has shape $(B, L, G \cdot d)$. Reshape to $(B, L, G, d)$, permute to $(B, G, L, d)$, then unsqueeze dim 2 to get $(B, G, 1, L, d)$. Show each step with concrete shapes. (2 points)
3. Explain how `Q @ K.mT` with shapes $(B, G, H/G, L, d)$ and $(B, G, 1, L, d)$ produces the correct logits via broadcasting. What is the output shape? (1 point)

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 8 (10 points, coding task)

**In this part, please build your own GQA module called `MyGQA`.**

- The requirement is similar to Part 4, with these differences:
  - The class name is `MyGQA`.
  - Constructor: `MyGQA(D, d, H, G)`.
  - Assert `H % G == 0` in `__init__`.
  - `W_Q`: `nn.Linear(D, H * d, bias=False)` — $H$ query heads.
  - `W_K`: `nn.Linear(D, G * d, bias=False)` — $G$ key groups (NOT $H$ heads).
  - `W_V`: `nn.Linear(D, G * d, bias=False)` — $G$ value groups.
  - `W_O`: `nn.Linear(H * d, D, bias=False)` — output projection.
- Method `forward`:
  - Input: `x` with shape `(B, L, D)` (self-attention: the same input serves as both attending and being attended sequence).
  - Output: shape `(B, L, D)`.
  - Use the 5D reshape + broadcasting strategy from Part 7.
  - Do NOT create $H/G$ copies of key/value projection matrices.
  - **No loop is allowed.**
  - After each operation, add a comment on the tensor shape.

### WRITE YOUR SOLUTION HERE ###

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 9 (5 points, coding task)

**Do the following tasks.**

Verify that `MyGQA` with $G = H$ produces the same output as `MyMHA`.

1. Create `MyGQA(D=32, d=8, H=4, G=4)`. (1 point)
2. Create `MyMHA(D=32, d=8, H=4)`. (1 point)
3. Copy the weights from the GQA model to the MHA model (when $G = H$, the shapes of `W_Q`, `W_K`, `W_V`, `W_O` are identical). (1 point)
4. Pass the same random input `x` of shape `(2, 5, 32)` through both: `model_mha(x, x)` and `model_gqa(x)`. (1 point)
5. Assert outputs match within tolerance `1e-5` using `torch.allclose`. (1 point)

### WRITE YOUR SOLUTION HERE ###

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

Now, let us study another variant of MHA: **Multi-head Latent Attention (MLA)**. MLA was introduced by **DeepSeek**. It is a core component of DeepSeek's large language model (LLM).

The key intuition of MLA is as follows. In MHA, the key and value projection matrices

$$\mathbf{W}^{\mathbf{K}, \text{MHA}} \in \mathbb{R}^{D \times D}, \quad \mathbf{W}^{\mathbf{V}, \text{MHA}} \in \mathbb{R}^{D \times D}$$

may be high dimensional.

For instance, suppose $D = 4096$.

However, it is not necessarily the case that these matrices are of high ranks (such as 4096). Their actual ranks (or top few ranks that make their truncated SVD close to the actual matrices) may be much lower than that.

To capture the low-rank feature, MLA proposes the following model:

$$\begin{aligned}
\mathbf{W}^{\mathbf{K}, \text{MHA}} & = {\color{blue}{\mathbf{W}^{\mathbf{UK}, \text{MLA}}}} \, {\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}} \\
\mathbf{W}^{\mathbf{V}, \text{MHA}} & = {\color{green}{\mathbf{W}^{\mathbf{UV}, \text{MLA}}}} \, {\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}}
\end{aligned}$$

where

- ${\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}} \in \mathbb{R}^{r \times D}$: shared down-projection matrix for computing keys and values.
- ${\color{blue}{\mathbf{W}^{\mathbf{UK}, \text{MLA}}}} \in \mathbb{R}^{D \times r}$: up-projection matrix for computing keys.
- ${\color{green}{\mathbf{W}^{\mathbf{UV}, \text{MLA}}}} \in \mathbb{R}^{D \times r}$: up-projection matrix for computing values.

In practice, rank $r$ is typically much smaller than $D$.

## Part 10 (10 points, non-coding task)

**In this part, you are asked to prove that GQA can be equivalently represented by MLA.**

In your solution, it is sufficient for you to prove that for $\mathbf{M} \in \left\{ \mathbf{K}, \mathbf{V} \right\}$, for the unrolled GQA matrix

$$\mathbf{\tilde W}^{\mathbf{M}, \text{GQA}} = \begin{bmatrix} {\color{orange}{\mathbf{W}^{\mathbf{M}, \text{GQA}}}} \\ {\color{orange}{\mathbf{W}^{\mathbf{M}, \text{GQA}}}} \\ \vdots \\ {\color{orange}{\mathbf{W}^{\mathbf{M}, \text{GQA}}}} \end{bmatrix} \in \mathbb{R}^{D \times D}$$

(the concatenation of $H/G$ copies of ${\color{orange}{\mathbf{W}^{\mathbf{M}, \text{GQA}}}} \in \mathbb{R}^{G \cdot d \times D}$),

the matrix $\mathbf{\tilde W}^{\mathbf{M}, \text{GQA}}$ can be decomposed as

$$\mathbf{\tilde W}^{\mathbf{M}, \text{GQA}} = {\color{blue}{\mathbf{W}^{\mathbf{UM}, \text{MLA}}}} \, {\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}}$$

where

- ${\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}} \in \mathbb{R}^{r \times D}$: down-projection matrix.
- ${\color{blue}{\mathbf{W}^{\mathbf{UM}, \text{MLA}}}} \in \mathbb{R}^{D \times r}$: up-projection matrix.
- $r = G \cdot d$.

**Reasoning is required.**

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 11 (5 points, non-coding task)

**In this part, you are asked to prove that GQA is NOT equivalent to MLA.** What you need to do is to find one example that MLA cannot be represented as GQA.

To be specific, please do the following things:

1. Construct ${\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}} \in \mathbb{R}^{1 \times 2}$.
2. Construct ${\color{blue}{\mathbf{W}^{\mathbf{UM}, \text{MLA}}}} \in \mathbb{R}^{2 \times 1}$.
3. Do matrix multiplication ${\color{blue}{\mathbf{W}^{\mathbf{UM}, \text{MLA}}}} \, {\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}}$.
4. Show that this product matrix is not the concatenation of two copies of a $1 \times 2$ matrix along axis 0. That is, the product is NOT in the form $\begin{bmatrix} a & b \\ a & b \end{bmatrix}$.

**Reasoning is required.**

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

MLA does not only enjoy its advantage of being more general than MHA and GQA, it is also computationally more efficient.

An intuitive approach of computing MLA:

1. Compute the key-projection matrix $\mathbf{W}^{\mathbf{UK}, \text{MLA}} \, \mathbf{W}^{\mathbf{DKV}, \text{MLA}} \in \mathbb{R}^{D \times D}$ and the value-projection matrix $\mathbf{W}^{\mathbf{UV}, \text{MLA}} \, \mathbf{W}^{\mathbf{DKV}, \text{MLA}} \in \mathbb{R}^{D \times D}$.
2. Follow the standard steps in MHA.

This approach is hereafter called a **vanilla approach**. This approach fails to enjoy the low-rank feature of ${\color{red}{\mathbf{W}^{\mathbf{DKV}, \text{MLA}}}}$, ${\color{blue}{\mathbf{W}^{\mathbf{UK}, \text{MLA}}}}$, and ${\color{green}{\mathbf{W}^{\mathbf{UV}, \text{MLA}}}}$.

## Part 12 (5 points, non-coding task)

**In this part, you are asked to study an alternative approach to compute MLA.**

1. Find a **head-independent reduced key-projection matrix** ${\color{red}{\hat{\mathbf{W}}^{\mathbf{K}, \text{MLA}}}} \in \mathbb{R}^{r \times D}$ and a **reduced query-projection matrix** ${\color{blue}{\hat{\mathbf{W}}^{\mathbf{Q}, \text{MLA}}}} \in \mathbb{R}^{H \cdot r \times D}$, such that
   - The **reduced key** at position $l_2$ is head-independent:
     $$\hat{\mathbf{k}}_{l_2} = {\color{red}{\hat{\mathbf{W}}^{\mathbf{K}, \text{MLA}}}} \, \mathbf{y}_{l_2} \in \mathbb{R}^r .$$
   - The **reduced query** at position $l_1$ for head $h$ is:
     $$\hat{\mathbf{q}}_{l_1, h} = {\color{blue}{\hat{\mathbf{W}}^{\mathbf{Q}, \text{MLA}}_h}} \, \mathbf{x}_{l_1} \in \mathbb{R}^r .$$
   - The attention score (query-key similarity) is invariant:
     $$\frac{\mathbf{q}_{l_1,h}^\top \mathbf{k}_{l_2,h}}{\sqrt{d}} = \frac{\hat{\mathbf{q}}_{l_1,h}^\top \hat{\mathbf{k}}_{l_2}}{\sqrt{r}} . \qquad (1)$$

2. Find a **head-independent reduced value-projection matrix** ${\color{green}{\hat{\mathbf{W}}^{\mathbf{V}, \text{MLA}}}} \in \mathbb{R}^{r \times D}$ and a **reduced out-projection matrix** ${\color{orange}{\hat{\mathbf{W}}^{O, \text{MLA}}}} \in \mathbb{R}^{D \times H \cdot r}$, such that the post-out-projection is invariant:
   $$\sum_{h=0}^{H-1} \mathbf{W}^O_h \sum_{l_2 = 0}^{L_2 - 1} \alpha_{h, l_1 l_2} \, \mathbf{v}_{l_2,h} = \sum_{h=0}^{H-1} {\color{orange}{\hat{\mathbf{W}}^{O, \text{MLA}}_h}} \sum_{l_2 = 0}^{L_2 - 1} \alpha_{h, l_1 l_2} \, \hat{\mathbf{v}}_{l_2} . \qquad (2)$$

Your answers for ${\color{red}{\hat{\mathbf{W}}^{\mathbf{K}, \text{MLA}}}}$, ${\color{blue}{\hat{\mathbf{W}}^{\mathbf{Q}, \text{MLA}}}}$, ${\color{green}{\hat{\mathbf{W}}^{\mathbf{V}, \text{MLA}}}}$, and ${\color{orange}{\hat{\mathbf{W}}^{O, \text{MLA}}}}$ should be written in terms of $\mathbf{W}^{\mathbf{DKV}}$, $\mathbf{W}^{\mathbf{UK}}$, $\mathbf{W}^{\mathbf{UV}}$, $\mathbf{W}^{\mathbf{Q}}$, and $\mathbf{W}^O$.

**Reasoning is required.**

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 13 (15 points, coding task)

**Do the following tasks.**

1. Define a function called `reduced_matrices`. (5 points)
   - Input arguments: `W_DKV`, `W_UK`, `W_UV`, `W_Q`, `W_O`, `H`.
   - Outputs: `W_K_MLA_hat`, `W_V_MLA_hat`, `W_Q_MLA_hat`, `W_O_MLA_hat`.
   - Requirement: the code for computing each output must be in **one line**. Loop is **not allowed**.

2. Set your device: (0 points)
   ```python
   device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
   ```

3. Construct the following synthetic data: (0 points)
   ```python
   D = 1024
   H = 32
   d = D // H
   r = 50
   
   W_DKV = torch.randn(r, D)
   W_UK = torch.randn(D, r)
   W_UV = torch.randn(D, r)
   W_Q = torch.randn(D, D)
   W_O = torch.randn(D, D)
   
   B = 32
   L_1 = 100
   L_2 = 300
   
   x = torch.randn(B, L_1, D).to(device)
   y = torch.randn(B, L_2, D).to(device)
   ```

4. Study a vanilla attention model: (5 points)
   - Initialize `model_MHA_vanilla = MyMHA(D, d, H)`.
   - Update weights: set `W_K.weight = W_UK @ W_DKV`, `W_V.weight = W_UV @ W_DKV`, `W_Q.weight = W_Q`, `W_O.weight = W_O`.
   - Compute `output_vanilla = model_MHA_vanilla(x, y)`.

5. Study a reduced attention model: (5 points)
   - Compute the reduced matrices using your function.
   - Initialize `model_MHA_reduced = MyMHA(D, r, H)` (note: `d` is replaced by `r`).
   - Set `W_K.weight = W_K_MLA_hat`, `W_V.weight = W_V_MLA_hat`, `W_Q.weight = W_Q_MLA_hat`, `W_O.weight = W_O_MLA_hat`.
   - Compute `output_reduced = model_MHA_reduced(x, y)`.
   - Print the relative error:
     ```python
     mse = nn.functional.mse_loss(output_reduced, output_vanilla)
     relative_error = mse ** 0.5 / torch.mean(output_vanilla ** 2) ** 0.5
     ```

### WRITE YOUR SOLUTION HERE ###

In [ ]:
### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

## Part 14 (5 points, non-coding task)

**Do the following tasks (Reasoning is required).**

In generative AI, such as GPT, we autoprogressively generate tokens. For a given position $l$, the keys and values on this position $\mathbf{k}_l$ and $\mathbf{v}_l$ are repeatedly used in generating tokens for positions $l' > l$.

Therefore, the values of $\mathbf{k}_l$ and $\mathbf{v}_l$ are typically stored in cache. We call such storage the **KV-cache**.

1. In MHA, the KV-cache at each position stores $\mathbf{k}_l \in \mathbb{R}^D$ and $\mathbf{v}_l \in \mathbb{R}^D$. What is the cache size per position per layer? (1 point)

2. In the reduced MLA (Part 12), the reduced key and reduced value are both equal to the compressed representation: $\hat{\mathbf{k}}_l = \hat{\mathbf{v}}_l = \mathbf{c}_l = {\color{red}{\hat{\mathbf{W}}^{\mathbf{K}, \text{MLA}}}} \, \mathbf{y}_l \in \mathbb{R}^r$. What is the MLA KV-cache at each position per layer? (1 point)

3. For $D = 4096, H = 32, d = 128, r = 512$: compute the MHA and MLA cache sizes per position per layer. What is the compression ratio? (1 point)

4. For a model with $N_L = 80$ layers, $L = 8192$ positions, FP16 (2 bytes per value): compute the total KV-cache in GB ($1 \text{ GB} = 2^{30}$ bytes) for both MHA and MLA. (2 points)

### WRITE YOUR SOLUTION HERE ###



""" END OF THIS PART """

---

**End of Problem 12.** Total: 100 points.

| Section | Parts | Points |
|---------|-------|--------|
| MHA Theory | 1--3 | 15 |
| MHA Implementation | 4--5 | 20 |
| GQA Theory | 6--7 | 10 |
| GQA Implementation + Verification | 8--9 | 15 |
| MLA Theory | 10--12 | 20 |
| MLA Implementation + KV-cache | 13--14 | 20 |
| **Total** | **1--14** | **100** |